In [3]:
import yfinance as yf
import pandas as pd


ttf = yf.download("TTF=F", start="2015-01-01", end = "2025-12-31")

print(ttf.head())

ModuleNotFoundError: No module named 'XGB'

In [24]:
df_ttf = ttf["Close"].copy().reset_index()
df_ttf.columns = ["Date", "pret_gas_TTF_euro"]
df_ttf["Date"] = pd.to_datetime(df_ttf["Date"])
df_ttf = df_ttf.resample("MS", on = "Date").mean().reset_index()
df_ttf["an"] = df_ttf["Date"].dt.year
df_ttf["luna"] = df_ttf["Date"].dt.month
df_ttf = df_ttf[["an", "luna", "pret_gas_TTF_euro"]]

print(df_ttf.head())

     an  luna  pret_gas_TTF_euro
0  2017    10          18.110714
1  2017    11          19.524048
2  2017    12          20.550750
3  2018     1          18.588334
4  2018     2          18.473421


In [25]:
df1 = pd.read_csv("eur_ron_lunar.csv")
df2 = pd.read_csv("preturi_energie_lunare.csv")
df3 = pd.read_csv("date_complete.csv")

df = pd.merge(df1, df2, on = ["an", "luna"], how = "inner")
df = pd.merge(df, df3, on = ["an", "luna"], how = "inner")
df = pd.merge(df, df_ttf, on = ["an", "luna"], how = "inner")

df["pret_gas_TTF_euro"] = df["pret_gas_TTF_euro"] * df["eur_ron_mediu"]

df["criza_energetica"] = (df["an"].isin([2022, 2023])).astype(int)

print(df[['an', 'luna', 'pret_energie_ron_mwh', 'pret_gas_TTF_euro']].head())

     an  luna  pret_energie_ron_mwh  pret_gas_TTF_euro
0  2017    10            216.599973          83.101506
1  2017    11            256.634194          90.424232
2  2017    12            174.757903          95.272022
3  2018     1            155.703387          86.435286
4  2018     2            176.874494          86.002736


In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

parametrii_test = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

xgboost_baza = XGBRegressor(random_state=42, objective = "reg:squarederror")

cautare = GridSearchCV(
    estimator = xgboost_baza,
    param_grid = parametrii_test,
    scoring = "r2",
    cv = 5,
    n_jobs = -1,
    verbose = 1
)
X = df.drop(columns=["an", "luna", "pret_energie_ron_mwh"])
Y = df["pret_energie_ron_mwh"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

cautare.fit(X_train, Y_train)
xgboost = cautare.best_estimator_
predictii = xgboost.predict(X_test)
r2 = r2_score(Y_test, predictii)
mae = mean_absolute_error(Y_test, predictii)

print(cautare.best_params_)
print(r2)
print(mae)

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/sorin__/PycharmProjects/ProiectPachete/.venv/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/sorin__/PycharmProjects/ProiectPachete/.venv/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Prebo

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="dark", rc={
"axes.facecolor": "#1c1c1c", "figure.facecolor": "#1c1c1c",
"text.color": "white", "axes.labelcolor": "white",
"xtick.color": "#bdbdbd", "ytick.color": "#bdbdbd",
"grid.color": "#2d2d2d", "axes.edgecolor": "#2d2d2d"
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=Y_test, y=predictii, color="#00bcd4", alpha=0.8, edgecolor="white", s=60, ax=axes[0])
max_val = max(Y_test.max(), predictii.max())
min_val = min(Y_test.min(), predictii.min())
axes[0].plot([min_val, max_val], [min_val, max_val], color='#e74c3c', linestyle='--', linewidth=2)

axes[0].set_title("Prețul Energiei: Valori Reale vs. Predicții", fontsize=14, fontweight='bold', color='white')
axes[0].set_xlabel("Preț Real (PZU)", fontsize=12)
axes[0].set_ylabel("Preț Prezis (Model)", fontsize=12)

importante = xgboost.feature_importances_
nume_coloane = X.columns

importanta_df = pd.DataFrame({'Variabilă': nume_coloane, 'Importanță': importante})
importanta_df = importanta_df.sort_values(by='Importanță', ascending=False)

sns.barplot(data=importanta_df, x='Importanță', y='Variabilă', palette="viridis", ax=axes[1])
axes[1].set_title("Ce influențează cel mai mult Prețul Energiei?", fontsize=14, fontweight='bold', color='white')
axes[1].set_xlabel("Scor de Importanță", fontsize=12)
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()